In [ ]:
# =============================================================================
# FUMD-AI Preprocessing Workflow -- Step 2: Add past-position lag columns to the SUMO trajectory
# =============================================================================
# Step:         2 of 7 (raw-input stage, before Step 3)
# Summary:      Add x-1..x-7 / y-1..y-7 lagged position columns to the base SUMO CSV, matching Step 4's expected input.
#
# Author(s):
#   - Cristina Bernad (ORCID: 0000-0001-9537-415X)
#   - Sonja Filiposka <sonja.filiposka@finki.ukim.mk> (ORCID: 0000-0003-0034-2855)
#   - Katja Gilly (ORCID: 0000-0002-8985-0639)
#
# Copyright:    (c) 2026 Cristina Bernad, Sonja Filiposka, Katja Gilly
# Repository:   https://github.com/FUMD-AI/fumd-ai-preprocessing-workflow
# Version:      1.0.2
# Funding:      This work has been funded by the FUMD-AI project, an EOSC GRAVITY -
#             Inter Project with Grant Number 25-EOSC-GRV-INTER-013.
#
# -----------------------------------------------------------------------------
# Licence
# Unless otherwise indicated:
#
#   * Source code in this notebook is licensed under the MIT License.
#
#   * Explanatory text and original figures are licensed under Creative
#     Commons Attribution 4.0 International (CC BY 4.0). Input datasets
#     retain the licences stated in their corresponding metadata or
#     source records.
#
# SPDX-License-Identifier: MIT
# -----------------------------------------------------------------------------
#
# Structured, machine-readable metadata for this workflow (authors, license,
# inputs/outputs per step) is also maintained in ro-crate-metadata.json at
# the repository root - update both together if either changes.
# =============================================================================


# Step 2 — Add past-position lag columns to the SUMO trajectory

Part of the **FUMD-AI preprocessing workflow**. Takes the base SUMO CSV
produced by `step_1_parse_raw_sumo_and_omnet.ipynb` (columns `t, veh_id,
x, y, angle, speed, pos, lane, slope, signals`) and adds, for each vehicle,
its position N seconds in the past for N in 1..7: `x-1..x-7`, `y-1..y-7`.

For vehicles that haven't been driving long enough yet to have N seconds
of history, the placeholder `999999` is used and **left as-is** here -
replacing it with a real coordinate happens later, in
`step_5_fix_past_positions.ipynb`, after this data has been merged with
the OMNeT++ feature matrix in Step 4. (An earlier draft of this notebook
replaced the placeholder immediately, which turned out to disagree with
the real reference data confirmed by round-tripping against it - the
placeholder genuinely needs to survive through Step 4.)

**Input:** `INPUT_PATH` - the base SUMO CSV from Step 1.

**Output:** `OUTPUT_PATH` - same columns, plus `x-1..x-7`/`y-1..y-7`
(with `999999` placeholders where a vehicle doesn't have that much
history yet). This is the file Step 4 expects as `VEHICLES_PATH`.

**Note:** the original version of this logic (in
`step_0_Vehicles_Classification.ipynb`) also computed *future* position
columns (`x1..x7`/`y1..y7`) and stop/move classification flags
(`c1..c7`/`c-1..c-7`). Those are not used anywhere downstream in this
workflow and are not reproduced here - this notebook only adds what
Step 4 actually needs. If you need the classification flags for your own
analysis, they can be re-derived the same way from `x1..x7`/`y1..y7`
before those are dropped.

**Known small discrepancy vs. the historical reference data.** This
notebook's logic was validated by round-tripping it against a real
`sumo_2_AI.csv`: stripping the lag columns back off, regenerating them
with the code below, and diffing against the original. The result matches
exactly for the overwhelming majority of rows, including every
"vehicle hasn't been driving long enough yet" sentinel case. However, a
small fraction of rows disagree (under 1% at `x-1`, growing to ~5% at
`x-7` in the validation sample) - specifically, rows where the vehicle has
spent more than ~1 second continuously on a SUMO internal junction lane
(lane id starting with `:`). The historical reference data has a sentinel
there; this notebook's simpler "look back N rows for this vehicle"
approach computes a real (plausible-looking) coordinate instead. Whatever
extra rule produced that historical behavior is not implemented in any
notebook currently in this workflow (it may be related to the
unused `boundaryXstart/Xend/Yend` coordinates seen in an earlier version
of `step_6_fix_past_positions.ipynb`, but this is speculative). If exact
bit-for-bit reproduction of the historical files matters for your use
case, investigate this further before relying on this notebook; otherwise
this is a reasonable, fully-documented simplification.


In [ ]:
import pandas as pd


In [ ]:
# ---- Parameters ----
# Defaults chain onto Step 1's default SUMO_OUTPUT_PATH, so this notebook
# runs out of the box straight after Step 1 using the bundled example.
INPUT_PATH = "sumo_trajectory_base.csv"   # Step 1 output (input)
OUTPUT_PATH = "sumo_trajectory.csv"       # past-position-augmented output (feeds Step 4's VEHICLES_PATH)

SENTINEL = 999999   # placeholder for "no history yet", replaced with the vehicle's first coordinate
N_LAG = 7           # number of lagged x-N/y-N column pairs to add
ROWS_PER_SECOND = 100  # sampling rate of the SUMO trajectory (e.g. 100 rows/s = 10ms step)


## 1. Load the base SUMO trajectory

In [ ]:
data = pd.read_csv(INPUT_PATH, delimiter="\t")
data["veh_id"] = data["veh_id"].astype(int)
print("input shape:", data.shape)
data.head()


## 2. Add lagged (past) position columns

`shift(+N)` looks backwards in time within each vehicle's own rows; rows
before a vehicle's first observation get the `SENTINEL` placeholder,
which is intentionally left in place - see Step 5 for where it gets
resolved to a real coordinate.


In [ ]:
for lag in range(1, N_LAG + 1):
    shift_rows = lag * ROWS_PER_SECOND
    data[f"x-{lag}"] = data.groupby("veh_id")["x"].shift(shift_rows).fillna(SENTINEL)
    data[f"y-{lag}"] = data.groupby("veh_id")["y"].shift(shift_rows).fillna(SENTINEL)

data.head()


## 3. Save the output

In [ ]:
data.to_csv(OUTPUT_PATH, index=False, sep="\t")
print(f"saved {OUTPUT_PATH} - shape {data.shape}")
data.head()
